# 03 — Model Training & Hyperparameter Tuning

**职责**: 加载处理后的数据 → 训练多个模型 → 超参数调优 → 保存最优模型

**架构**: 每位成员在 `src/models/` 下维护自己的模型文件，notebook 自动发现并执行。  
**输入**: `data/processed/train_processed.csv`  
**输出**: `outputs/models/*.pkl`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from config import DATA_PROC, TARGET, MODEL_DIR
from src.models import discover_models, build_models, save_model
from src.evaluate import cross_validate_all, results_to_dataframe

## 1. 加载处理后的数据

In [ ]:
df = pd.read_csv(DATA_PROC)
X = df.drop(columns=[TARGET])
y = df[TARGET]
print(f"X: {X.shape}, y: {y.shape}")
print(f"Features: {list(X.columns)}")

## 2. 自动发现模型 & 交叉验证

`discover_models()` 会自动扫描 `src/models/` 下所有 `BaseModel` 子类。  
新增模型只需创建文件并继承 `BaseModel`，无需修改此 notebook。

In [ ]:
registry = discover_models()
print(f"Discovered {len(registry)} models: {list(registry.keys())}\n")

pipelines = {name: m.build_pipeline() for name, m in registry.items()}

results = cross_validate_all(pipelines, X, y)
results_to_dataframe(results)

## 3. 超参数调优

对表现最好的模型做 GridSearch / RandomSearch 微调。

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

best_model_name = results_to_dataframe(results).index[0]
best_entry = registry[best_model_name]
grid = best_entry.param_grid()

if grid:
    print(f"Tuning: {best_model_name}")
    print(f"Param grid: {grid}\n")
    search = RandomizedSearchCV(
        best_entry.build_pipeline(), grid,
        n_iter=20, cv=5, scoring='neg_mean_squared_error',
        n_jobs=-1, random_state=42, verbose=1,
    )
    search.fit(X, y)
    print(f"\nBest params: {search.best_params_}")
    print(f"Best RMSE: {np.sqrt(-search.best_score_):.3f}")
    pipelines[best_model_name] = search.best_estimator_
else:
    print(f"{best_model_name} has no param_grid defined, skipping tuning.")

## 4. 保存模型

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name, pipe in pipelines.items():
    pipe.fit(X, y)
    save_model(pipe, name)
    print(f"Saved: {name}")